In [16]:
import socket                                        
import time                                            
import re                                         
import requests                                    
from datetime import datetime, timedelta               
from confluent_kafka import Producer, Consumer        
from confluent_kafka.admin import AdminClient, NewTopic  

# Address of the Kafka server (the Docker container)
BOOTSTRAP = 'localhost:9092'

In [ ]:
##P LAB

In [27]:
#create a new topic called book

futures = admin_client.create_topics([NewTopic('book', num_partitions=1, replication_factor=1)])

for topic_name, future in futures.items():
    try:
        future.result()                                
        print(f"Topic '{topic_name}' created")
    except Exception as e:
        print(f"Topic '{topic_name}' not created: {e}")  

print(list(admin_client.list_topics().topics.keys()))  # check that book is listed

Topic 'book' not created: KafkaError{code=TOPIC_ALREADY_EXISTS,val=36,str="Topic 'book' already exists."}
['timer', 'book', '__consumer_offsets']


In [28]:
#download a book from Project Gutenberg and save it in a local file 

url = "https://www.gutenberg.org/cache/epub/1342/pg1342.txt"  # Pride and Prejudice
response = requests.get(url)                          

with open("book.txt", "w", encoding="utf-8") as f:   
    f.write(response.text)

print("Book downloaded")

Book downloaded


In [29]:
# read book.txt line by line and send each line to the book topic 

book_producer = Producer({'bootstrap.servers': BOOTSTRAP})

with open("book.txt", "r", encoding="utf-8") as f:
    for line in f:                                     # one line at a time
        if line.strip():                               
            book_producer.produce(topic='book', value=line.strip())  
            book_producer.poll(0)                      

book_producer.flush()                                  
print("Book sent")

Book sent


In [30]:
# read the book topic clean each line like in the word count lab,
# and write the result in cleaned_book.txt 

stop_words = {'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for',
              'from', 'has', 'he', 'in', 'is', 'it', 'its', 'of', 'on',
              'or', 'that', 'the', 'to', 'was', 'will', 'with'}


punctuation = ".,!?;:'-\"()[]{}—–"

def clean_line(line):
    """Lowercase, split into words remove punctuation, blanks and stop words."""
    words = line.lower().split(' ')                    
    words = [re.sub(f'[{re.escape(punctuation)}]', '', w) for w in words]  
    words = [w for w in words if w.strip() != '' and w not in stop_words]  
    return words

book_consumer = Consumer({'bootstrap.servers': BOOTSTRAP,
                          'group.id': 'book_group',    
                          'auto.offset.reset': 'smallest'})  
book_consumer.subscribe(['book'])                      

empty_polls = 0                                        
with open("cleaned_book.txt", "w", encoding="utf-8") as f:  
    while empty_polls < 10:                            
        msg = book_consumer.poll(1.0)                  
        if msg is None:
            empty_polls += 1
            continue
        if msg.error():
            print(f"Consumer error: {msg.error()}")
            continue
        empty_polls = 0                              
        words = clean_line(msg.value().decode('utf-8'))  
        if words:                                     
            f.write(' '.join(words) + "\n")            

book_consumer.close()                                  
print("Done, output in cleaned_book.txt")

Done, output in cleaned_book.txt


In [31]:
# display the first 500 characters of the cleaned file to check the result

print(open("cleaned_book.txt", encoding="utf-8").read()[:500])

project gutenberg ebook pride prejudice
this ebook use anyone anywhere united states
most other parts world no cost almost no restrictions
whatsoever you may copy give away reuse under terms
project gutenberg license included this ebook online
wwwgutenbergorg if you not located united states
you have check laws country where you located
before using this ebook
title pride prejudice
author jane austen
release date june 1 1998 ebook #1342
most recently updated september 1 2026
language english
oth
